# ACML Assignment 2: Convolutional Autoencoders

**Authors:** Simon Delecluse, Francesco xxxx

Goal: build a convolutional autoencoder (CAE) on CIFAR-10, study the latent space and architecture variations, then use it to colourise grayscale images.

## 0. Setup

In [18]:
import time
import copy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

import torchvision
import torchvision.transforms as T

from skimage.color import rgb2lab, lab2rgb  # for the chrominance (Lab) trick

In [19]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 50          # upper bound, early stopping usually ends sooner (PDF asks for >= 10)
LR = 1e-3
PATIENCE = 5         # early stopping: epochs without val improvement before stopping
LR_PATIENCE = 2      # scheduler: epochs without val improvement before lowering the LR
LR_FACTOR = 0.5      # scheduler: multiply LR by this factor
print(DEVICE)

cpu


## 1. Data
CIFAR-10 (60 000 colour images, 32x32x3). As suggested in the PDF we use `torchvision.datasets.CIFAR10`.
The official train (50k) and test (10k) sets are merged, shuffled and re-split **80 / 10 / 10** into train / validation / test (48k / 6k / 6k).
Pixels are normalised from [0, 255] to **[0, 1]**, which matches the sigmoid output of the decoder.

In [ ]:
# Load CIFAR-10: merge the official train (50k) and test (10k) sets -> 60k images
train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)
x = np.concatenate([train_set.data, test_set.data])     # (60000, 32, 32, 3), uint8
x = torch.from_numpy(x).permute(0, 3, 1, 2)              # -> (N, C, H, W) as PyTorch expects

# Normalise from [0, 255] to [0, 1]
x = x.float() / 255.0

# Shuffle and split 80 / 10 / 10
n = len(x)
idx = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
n_train, n_val = int(0.8 * n), int(0.1 * n)
x_train = x[idx[:n_train]]
x_val = x[idx[n_train:n_train + n_val]]
x_test = x[idx[n_train + n_val:]]

# DataLoaders: input = target for reconstruction
train_loader = DataLoader(TensorDataset(x_train, x_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, x_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(x_test, x_test), batch_size=BATCH_SIZE)

print("all  :", tuple(x.shape), f"range [{x.min():.1f}, {x.max():.1f}]")
print("train:", tuple(x_train.shape))
print("val  :", tuple(x_val.shape))
print("test :", tuple(x_test.shape))

  1%|▏         | 2.49M/170M [00:25<28:36, 97.9kB/s]


KeyboardInterrupt: 

## 2. Helpers: training, evaluation, plotting
Training uses **Adam** with **MSE** loss, plus:
- **Learning-rate scheduling** (`ReduceLROnPlateau`): when the validation loss stops improving for `LR_PATIENCE` epochs, the LR is multiplied by `LR_FACTOR`.
- **Early stopping**: if the validation loss has not improved for `PATIENCE` epochs, training stops and the weights from the best epoch are restored.

In [ ]:
def evaluate(model, loader, loss_fn=None):
    """Average loss over a loader (used for val and test error)."""
    loss_fn = loss_fn or nn.MSELoss()
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            total += loss_fn(model(xb), yb).item() * len(xb)
            n += len(xb)
    return total / n


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, loss_fn=None,
                patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR, verbose=True):
    """Train with Adam + MSE, LR scheduling and early stopping.
    Returns history dict with train_loss, val_loss and lr per epoch, plus best_epoch."""
    loss_fn = loss_fn or nn.MSELoss()
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                           factor=lr_factor, patience=lr_patience)
    history = {"train_loss": [], "val_loss": [], "lr": []}

    # early stopping state
    best_loss = float("inf")
    best_state = None
    best_epoch = 0
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        total, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)
            n += len(xb)

        train_loss = total / n
        val_loss = evaluate(model, val_loader, loss_fn)
        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        if verbose:
            print(f"epoch {epoch:3d} | train {train_loss:.5f} | val {val_loss:.5f} "
                  f"| lr {current_lr:.1e} | {time.time() - t0:.1f}s")

        scheduler.step(val_loss)

        # early stopping: keep the best weights, stop after `patience` epochs without improvement
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch} (best epoch {best_epoch}, val {best_loss:.5f})")
                break

    model.load_state_dict(best_state)   # restore the best weights
    history["best_epoch"] = best_epoch
    return history


def plot_learning_curve(history, title=""):
    """Train and val loss over epochs, with the learning rate on a second axis."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, history["train_loss"], label="train")
    ax.plot(epochs, history["val_loss"], label="validation")
    if "best_epoch" in history:
        ax.axvline(history["best_epoch"], color="gray", ls="--", lw=1, label="best epoch")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE")
    ax.set_title(title)

    ax2 = ax.twinx()
    ax2.plot(epochs, history["lr"], color="tab:red", alpha=0.4, ls=":", label="learning rate")
    ax2.set_ylabel("learning rate")
    ax2.set_yscale("log")

    lines = ax.get_legend_handles_labels()
    lines2 = ax2.get_legend_handles_labels()
    ax.legend(lines[0] + lines2[0], lines[1] + lines2[1])
    plt.tight_layout()
    plt.show()


def show_reconstructions(model, x, n=8, title=""):
    """Originals on top row, reconstructions below."""
    model.eval()
    with torch.no_grad():
        rec = model(x[:n].to(DEVICE)).cpu()
    fig, axes = plt.subplots(2, n, figsize=(1.5 * n, 3.2))
    for i in range(n):
        axes[0, i].imshow(x[i].permute(1, 2, 0).numpy())
        axes[1, i].imshow(rec[i].clamp(0, 1).permute(1, 2, 0).numpy())
        axes[0, i].axis("off")
        axes[1, i].axis("off")
    axes[0, 0].set_title("original", loc="left", fontsize=9)
    axes[1, 0].set_title("reconstruction", loc="left", fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def latent_size(model, input_shape=(1, 3, 32, 32)):
    """Run a dummy input through the encoder and return the latent shape and number of elements."""
    with torch.no_grad():
        z = model.encode(torch.zeros(input_shape, device=next(model.parameters()).device))
    return tuple(z.shape[1:]), z[0].numel()

## 3. Exercise 1: Reconstruction with the given CAE
Architecture from the PDF (padding "same", stride 1 for convs, ReLU):

`Conv 3x3 (8) -> MaxPool 2x2 -> Conv 3x3 (12) -> MaxPool 2x2 -> Conv 3x3 (16) -> Upsample 2x2 -> Conv 3x3 (12) -> Upsample 2x2 -> Conv 3x3 (3)`

**Stride** is the step (in pixels) the filter moves between two positions.
- Conv layers use **stride 1**: the 3x3 filter is applied at every pixel, so with padding 1 the spatial size stays 32 -> 32.
- Pooling uses **stride 2** (the default, equal to the 2x2 window): windows don't overlap, so the size halves 32 -> 16 -> 8.
- Upsampling 2x2 does the reverse, repeating each pixel into a 2x2 block: 8 -> 16 -> 32.

The **encoder** (conv, pool, conv, pool, conv) compresses the image to a latent code of 8x8x16. The **decoder** (upsample, conv, upsample, conv) reconstructs the 32x32x3 image from it.
The last layer uses a **sigmoid** so the output is in [0, 1], like the normalised input.

In [ ]:
class CAE(nn.Module):
    """Baseline convolutional autoencoder from the assignment."""

    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, stride=1, padding=1),    # 32x32x8
            nn.ReLU(),
            nn.MaxPool2d(2),                                       # 16x16x8
            nn.Conv2d(8, 12, kernel_size=3, stride=1, padding=1),   # 16x16x12
            nn.ReLU(),
            nn.MaxPool2d(2),                                       # 8x8x12
            nn.Conv2d(12, 16, kernel_size=3, stride=1, padding=1),  # 8x8x16  <- latent
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2),                            # 16x16x16
            nn.Conv2d(16, 12, kernel_size=3, stride=1, padding=1),  # 16x16x12
            nn.ReLU(),
            nn.Upsample(scale_factor=2),                            # 32x32x12
            nn.Conv2d(12, 3, kernel_size=3, stride=1, padding=1),   # 32x32x3
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        return self.decode(self.encode(x))


model = CAE().to(DEVICE)
print(model)
print("parameters:", count_params(model))
print("latent:", latent_size(model))

In [ ]:
history = train_model(model, train_loader, val_loader)
plot_learning_curve(history, "Baseline CAE")
print("Test MSE:", evaluate(model, test_loader))
show_reconstructions(model, x_test, title="Baseline CAE (test set)")

**Discussion:** _learning curve, test error, quality of reconstructions._

## 4. Exercise 2: Latent space and architecture variations
### 4.1 Latent space size of the baseline
Using $\left(\frac{W-K+2P}{S}+1\right)^2 \times C$ per layer.

In [ ]:
def conv_output_size(W, K, P, S):
    """Spatial size after a conv/pool layer."""
    pass


def layer_sizes_table(model, input_shape=(1, 3, 32, 32)):
    """Print the output shape and size of every layer (to check the hand calculation)."""
    pass

In [ ]:
# layer_sizes_table(CAE())

**Answer:** _latent size = ..._

### 4.2 Other architectures

In [ ]:
class ConfigurableCAE(nn.Module):
    """CAE built from a config: list of channels per encoder block, kernel size, stride, pooling vs strided conv."""

    def __init__(self, channels=(8, 12, 16), kernel_size=3, stride=1, use_pooling=True, in_ch=3, out_ch=3):
        super().__init__()
        pass

    def forward(self, x):
        pass

In [ ]:
ARCHITECTURES = {
    # "baseline": dict(channels=(8, 12, 16)),
    # "wider":    dict(channels=(32, 64, 128)),
    # "shallow":  dict(channels=(16, 32)),
    # "kernel5":  dict(channels=(8, 12, 16), kernel_size=5),
    # "strided":  dict(channels=(8, 12, 16), use_pooling=False, stride=2),
}


def run_experiments(architectures, train_loader, val_loader, test_loader, epochs=EPOCHS):
    """Train each config, return a results table: name, latent size, params, test MSE, history."""
    pass


def plot_latent_vs_error(results):
    """Scatter of latent size vs test error."""
    pass

In [ ]:
# results = run_experiments(ARCHITECTURES, train_loader, val_loader, test_loader)
# plot_latent_vs_error(results)

**Discussion:** _impact of each change; correlation between latent size and error._

## 5. Exercise 3: Colourisation
### 5.1 Grayscale -> RGB
Input: 1-channel grayscale, target: 3-channel RGB. Start from the best architecture of Exercise 2.

In [ ]:
def to_grayscale(x):
    """(N, 3, H, W) RGB -> (N, 1, H, W) luminance."""
    pass


def show_colourisation(model, x_gray, x_true, n=8, title=""):
    """Rows: grayscale input, prediction, ground truth."""
    pass

In [ ]:
# x_train_g, x_val_g, x_test_g = map(to_grayscale, (x_train, x_val, x_test))
# col_train_loader = DataLoader(TensorDataset(x_train_g, x_train), batch_size=BATCH_SIZE, shuffle=True)
# col_val_loader = DataLoader(TensorDataset(x_val_g, x_val), batch_size=BATCH_SIZE)
# col_test_loader = DataLoader(TensorDataset(x_test_g, x_test), batch_size=BATCH_SIZE)
# col_model = ConfigurableCAE(**best_config, in_ch=1, out_ch=3).to(DEVICE)

### 5.2 Chrominance trick (Lab colour space)
Feed L, predict a/b, then merge back with the known L.

In [ ]:
def rgb_to_lab(x):
    """(N, 3, H, W) RGB in [0,1] -> L in [0,1] (N,1,H,W) and ab scaled to [-1,1] (N,2,H,W)."""
    pass


def lab_to_rgb(L, ab):
    """Inverse of rgb_to_lab, returns RGB in [0,1]."""
    pass

In [ ]:
# L_*, ab_* = rgb_to_lab(...)
# lab_model = ConfigurableCAE(**best_config, in_ch=1, out_ch=2).to(DEVICE)

### 5.3 Improvements
_Ideas to try: deeper/wider network, skip connections (U-Net), batch norm, different loss, more epochs, LR schedule._

In [ ]:
# improved model(s) here

**Discussion:** _results, limitations (blurry / desaturated colours, dataset bias), possible improvements._

## 6. Conclusion